# 05 - Feature-based model (XGBoost)

Covariate engineering (Part 5) + tuned XGBoost with genuinely recursive 24 h forecasts (Part 6). Two variants:
- `feature_model`: all covariates (conditional forecast)
- `feature_model_pastonly`: lags/rolling/time only (true operational forecast)

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from appliance_energy.config import *
from appliance_energy import data, eda, features, evaluation, plotting
from appliance_energy.models import feature_models as fm

In [ ]:
hourly = data.resample_hourly(data.load_raw())
y = hourly[TARGET]
train, test = data.train_test_split(y, TEST_STEPS)
table = features.build_feature_matrix(hourly)
print(table.shape)
list(table.columns)

Hyperparameters were tuned on the last 14 days of the *training* period (never the test set):

In [ ]:
pd.read_csv(METRICS_DIR / 'xgb_tuning_feature_model.csv').head()

In [ ]:
cols = features.feature_columns(table, include_covariates=True)
tr = table.loc[table.index < test.index[0]]
model = fm.fit_xgb(tr[cols], tr[TARGET])
fc = fm.recursive_forecast(model, table, cols, y, test.index, HORIZON)
evaluation.metrics_table([
    evaluation.evaluate_forecast('feature_model', test, fc, train)])

In [ ]:
grouped = fm.grouped_importance(model, cols, features.feature_group)
plotting.plot_grouped_importance(grouped, 'fig09_feature_importance.png')
display(Image(str(FIGURE_DIR / 'fig09_feature_importance.png')))

Careful with the grouped importances: the indoor-sensor group sums highest only because it has 19 features. Per feature, lags dominate (lag_1 alone is the top feature), then time and rolling features; outdoor weather is last. The past-only variant generalises *better* (see notebook 07) - the sensor covariates mostly add noise at a 24 h recursive horizon.